In [1]:
import argparse
import csv
import json
import os
import re
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path
from typing import List, Optional

import pandas as pd
import penman
import torch
from torch.utils.data import Dataset
from tqdm import tqdm
from transformers import (
    DataCollatorForSeq2Seq,
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
from transformers.utils import logging as hf_logging

In [2]:
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

In [3]:
MODEL_NAME = "BramVanroy/mbart-large-cc25-ft-amr30-en"
SRC_LANG = "hi_IN"  # Closest mBART language code for Konkani
TGT_LANG = "en_XX"
MAX_SRC_LEN = 128
MAX_TGT_LEN = 512
SEED = 42

In [4]:
# ─────────────────────────────────────────────
# AMR helpers  (identical to your inference script)
# ─────────────────────────────────────────────


def linearized_to_penman(linearized: str, graph_idx: int = 0) -> str:
    """Convert linearized MBart AMR output → Penman string."""
    text = linearized.replace("<AMR>", "").strip()
    pointer_map: dict = {}
    var_idx = 0

    def replace_pointer(match):
        nonlocal var_idx
        idx = match.group(1)
        if idx not in pointer_map:
            pointer_map[idx] = f"x{graph_idx}_{var_idx}"
            var_idx += 1
        return f"({pointer_map[idx]} / "

    text = re.sub(r"<pointer:(\d+)>", replace_pointer, text)
    text = text.replace("<rel>", " ").replace("</rel>", ")")
    text = re.sub(r"\s+", " ", text).strip()

    open_par = text.count("(")
    close_par = text.count(")")
    text += ")" * max(0, open_par - close_par)
    return text


def clean_pred_penman(text: str) -> str:
    """Clean up common artefacts in predicted Penman strings."""
    text = re.sub(
        r"<lit>\s*(.*?)\s*</lit>",
        lambda m: '"' + m.group(1).strip() + '"',
        text,
        flags=re.DOTALL,
    )
    text = re.sub(r"^thing\(", "(", text.strip())
    text = re.sub(r"\(x\d+_\d+\s*/\s*\)", "(amr-unknown)", text)
    text = re.sub(
        r"(:(?:ARG\d+|op\d+|mod|poss|quant|domain|time|location|manner|"
        r"cause|degree|purpose|condition|wiki|name|polarity|mode|li|value|snt\d+))"
        r"(x\d+_\d+)",
        r"\1 \2",
        text,
    )
    return text


def safe_encode(amr_str: str) -> Optional[str]:
    try:
        return penman.encode(penman.decode(amr_str.strip()))
    except Exception:
        return None


def smatch_score(gold_str: str, pred_str: str):
    """Return (P, R, F1) or None on failure."""
    gold_norm = safe_encode(gold_str)
    pred_norm = safe_encode(pred_str)
    if gold_norm is None or pred_norm is None:
        return None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".amr", delete=False, encoding="utf-8"
        ) as gf:
            gf.write(gold_norm + "\n\n")
            gold_path = gf.name
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".amr", delete=False, encoding="utf-8"
        ) as pf:
            pf.write(pred_norm + "\n\n")
            pred_path = pf.name

        result = subprocess.run(
            [sys.executable, "-m", "smatch", "--pr", "-f", pred_path, gold_path],
            capture_output=True,
            text=True,
            timeout=15,
        )
        os.unlink(gold_path)
        os.unlink(pred_path)

        p = r = f = None
        for line in result.stdout.strip().split("\n"):
            if "Precision" in line:
                p = float(line.split()[-1])
            elif "Recall" in line:
                r = float(line.split()[-1])
            elif "F-score" in line:
                f = float(line.split()[-1])
        if f is not None:
            return round(p, 4), round(r, 4), round(f, 4)
        return None
    except Exception:
        return None

In [5]:
class KonkaniAMRDataset(Dataset):
    def __init__(
        self,
        sentences: List[str],
        amr_strings: List[str],
        tokenizer: MBart50TokenizerFast,
    ):
        self.sentences = sentences
        self.amr_strings = amr_strings
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        src = self.sentences[idx]
        tgt = self.amr_strings[idx]

        self.tokenizer.src_lang = SRC_LANG
        model_inputs = self.tokenizer(
            src,
            max_length=MAX_SRC_LEN,
            truncation=True,
            padding=False,
        )

        # with self.tokenizer.as_target_tokenizer():
        #     labels = self.tokenizer(
        #         tgt,
        #         max_length=MAX_TGT_LEN,
        #         truncation=True,
        #         padding=False,
        #     )
        # Change this:
        labels = self.tokenizer(
            text_target=tgt,
            max_length=MAX_TGT_LEN,
            truncation=True,
            padding=False,
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


# ─────────────────────────────────────────────
# Data split
# ─────────────────────────────────────────────


def split_data(df: pd.DataFrame, train_frac=0.80, val_frac=0.05, seed=SEED):
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n = len(df)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train = df.iloc[:n_train]
    val = df.iloc[n_train : n_train + n_val]
    test = df.iloc[n_train + n_val :]
    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )


# ─────────────────────────────────────────────
# Inference on a list of sentences
# ─────────────────────────────────────────────


def run_inference(
    sentences: List[str],
    gold_amrs: List[str],
    model: MBartForConditionalGeneration,
    tokenizer: MBart50TokenizerFast,
    device: str,
    batch_size: int = 8,
) -> List[dict]:
    model.eval()
    results = []
    forced_bos = tokenizer.lang_code_to_id[TGT_LANG]

    for start in tqdm(range(0, len(sentences), batch_size), desc="Inference"):
        batch_sents = sentences[start : start + batch_size]
        batch_golds = gold_amrs[start : start + batch_size]

        tokenizer.src_lang = SRC_LANG
        inputs = tokenizer(
            batch_sents,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SRC_LEN,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                max_length=MAX_TGT_LEN,
                num_beams=4,
                early_stopping=True,
            )

        raw_outputs = tokenizer.batch_decode(generated, skip_special_tokens=True)

        for i, (sent, gold, raw) in enumerate(
            zip(batch_sents, batch_golds, raw_outputs)
        ):
            global_idx = start + i
            pred_penman = clean_pred_penman(
                linearized_to_penman(raw, graph_idx=global_idx)
            )
            results.append(
                {
                    "sentence": sent,
                    "gold_amr": gold.strip(),
                    "model_output_linearized": raw,
                    "model_output_penman": pred_penman,
                }
            )
    return results


# ─────────────────────────────────────────────
# Smatch evaluation
# ─────────────────────────────────────────────


def evaluate_smatch(predictions: List[dict], output_csv: str) -> None:
    results_out = []
    scored = skipped = 0

    for i, item in enumerate(tqdm(predictions, desc="Smatch")):
        score = smatch_score(item["gold_amr"], item["model_output_penman"])
        if score is None:
            skipped += 1
            results_out.append(
                {
                    "idx": i,
                    "sentence": item["sentence"],
                    "P": "",
                    "R": "",
                    "F1": "",
                    "status": "SKIP",
                }
            )
        else:
            p, r, f1 = score
            scored += 1
            results_out.append(
                {
                    "idx": i,
                    "sentence": item["sentence"],
                    "P": p,
                    "R": r,
                    "F1": f1,
                    "status": "OK",
                }
            )

    ok = [r for r in results_out if r["status"] == "OK"]
    f1_vals = [r["F1"] for r in ok]
    p_vals = [r["P"] for r in ok]
    r_vals = [r["R"] for r in ok]

    print(f"\n{'=' * 55}")
    print(f"Total test examples : {len(predictions)}")
    print(f"Scored (parsed)     : {scored}")
    print(f"Skipped             : {skipped}")
    if f1_vals:
        print(f"\n--- Scores on {scored} parseable pairs ---")
        print(f"Avg Precision   : {sum(p_vals) / len(p_vals):.4f}")
        print(f"Avg Recall      : {sum(r_vals) / len(r_vals):.4f}")
        print(f"Avg F1          : {sum(f1_vals) / len(f1_vals):.4f}")
        all_f1 = f1_vals + [0.0] * skipped
        print(f"\n--- Treating skipped as F1=0 (n={len(predictions)}) ---")
        print(f"Avg F1 (all)    : {sum(all_f1) / len(all_f1):.4f}")
        print("\nTop-10 F1:")
        for r in sorted(ok, key=lambda x: -x["F1"])[:10]:
            print(f"  [{r['idx']:04d}] F1={r['F1']:.3f}  {r['sentence'][:55]}")

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["idx", "sentence", "P", "R", "F1", "status"])
        w.writeheader()
        w.writerows(results_out)
    print(f"\nSaved smatch scores → {output_csv}")

In [8]:
set_seed(42)

output_dir = Path('./konkani_amr_finetuned')
output_dir.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [11]:
# ── 1. Load & split data ──────────────────────────────────────────────
df = pd.read_csv(r'data.csv')
assert "sentence" in df.columns and "amr_penman" in df.columns, (
    "CSV must have 'sentence' and 'amr_penman' columns."
)
df = df.dropna(subset=["sentence", "amr_penman"]).reset_index(drop=True)
print(f"Total examples after dropping NaN: {len(df)}")

train_df, val_df, test_df = split_data(df, seed=42)
print(f"Split  →  train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

# Save splits for reproducibility
train_df.to_csv(output_dir / "train.csv", index=False)
val_df.to_csv(output_dir / "val.csv", index=False)
test_df.to_csv(output_dir / "test.csv", index=False)
print(f"Splits saved to {output_dir}/{{train,val,test}}.csv")

Total examples after dropping NaN: 1100
Split  →  train: 880  val: 55  test: 165
Splits saved to konkani_amr_finetuned/{train,val,test}.csv


In [17]:
# ── 2. Tokenizer & model ──────────────────────────────────────────────
tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(device)

tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG

In [18]:
train_dataset = KonkaniAMRDataset(
    train_df["sentence"].tolist(), train_df["amr_penman"].tolist(), tokenizer
)
val_dataset = KonkaniAMRDataset(
    val_df["sentence"].tolist(), val_df["amr_penman"].tolist(), tokenizer
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)

In [19]:
# ── 4. Training arguments ─────────────────────────────────────────────
model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id[TGT_LANG]
training_args = Seq2SeqTrainingArguments(
    output_dir=str(output_dir / "checkpoints"),
    num_train_epochs=20,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=False,  # loss-only eval during training
    fp16=True and device == "cuda",
    logging_dir=str(output_dir / "logs"),
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=42,
)

In [20]:
%%time
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    # tokenizer=tokenizer,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("\nStarting fine-tuning …")
trainer.train()


Starting fine-tuning …
{'loss': 5.7273, 'grad_norm': 3.2916183471679688, 'learning_rate': 2.45e-05, 'epoch': 0.9090909090909091}
{'eval_loss': 2.486176013946533, 'eval_runtime': 0.4131, 'eval_samples_per_second': 133.154, 'eval_steps_per_second': 33.894, 'epoch': 1.0}
{'loss': 2.2063, 'grad_norm': 2.5880322456359863, 'learning_rate': 4.9500000000000004e-05, 'epoch': 1.8181818181818183}
{'eval_loss': 1.7850672006607056, 'eval_runtime': 0.3786, 'eval_samples_per_second': 145.269, 'eval_steps_per_second': 36.978, 'epoch': 2.0}
{'loss': 1.6227, 'grad_norm': 2.2106289863586426, 'learning_rate': 4.755e-05, 'epoch': 2.7272727272727275}
{'eval_loss': 1.5876933336257935, 'eval_runtime': 0.374, 'eval_samples_per_second': 147.053, 'eval_steps_per_second': 37.432, 'epoch': 3.0}
{'loss': 1.2712, 'grad_norm': 2.2405731678009033, 'learning_rate': 4.5050000000000004e-05, 'epoch': 3.6363636363636362}
{'eval_loss': 1.5330950021743774, 'eval_runtime': 0.3753, 'eval_samples_per_second': 146.566, 'eval_st

TrainOutput(global_step=1100, training_loss=1.6180211023850875, metrics={'train_runtime': 493.9014, 'train_samples_per_second': 35.635, 'train_steps_per_second': 2.227, 'train_loss': 1.6180211023850875, 'epoch': 20.0})

In [21]:
%%time
# Save the best model + tokenizer in a clean location
best_model_dir = str(output_dir / "best_model")
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print(f"Best model saved → {best_model_dir}")

Best model saved → konkani_amr_finetuned/best_model
CPU times: user 729 ms, sys: 1.66 s, total: 2.39 s
Wall time: 2.46 s


In [24]:
%%time
# ── 6. Reload best model for inference ───────────────────────────────
print("\nLoading best model for test inference …")
model = MBartForConditionalGeneration.from_pretrained(best_model_dir)
model.to(device)
# tokenizer = MBart50TokenizerFast.from_pretrained(best_model_dir)
tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG

# ── 7. Inference on test set ──────────────────────────────────────────
test_sentences = test_df["sentence"].tolist()
test_gold_amrs = test_df["amr_penman"].tolist()

predictions = run_inference(
    test_sentences,
    test_gold_amrs,
    model,
    tokenizer,
    device,
    batch_size=8,
)

pred_json_path = str(output_dir / "test_predictions.json")
with open(pred_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)
print(f"\nTest predictions saved → {pred_json_path}")

# ── 8. Smatch evaluation ──────────────────────────────────────────────
smatch_csv_path = str(output_dir / "smatch_scores_test.csv")
evaluate_smatch(predictions, smatch_csv_path)


Loading best model for test inference …


Inference: 100%|██████████| 21/21 [00:52<00:00,  2.52s/it]



Test predictions saved → konkani_amr_finetuned/test_predictions.json


Smatch:   0%|          | 0/165 [00:00<?, ?it/s]ignoring epigraph data for duplicate triple: ('p3', ':instance', 'place')
ignoring secondary node contexts for 'p3'
ignoring secondary node contexts for 'n5'
ignoring secondary node contexts for 'p4'
Smatch:   7%|▋         | 12/165 [00:00<00:08, 17.78it/s]ignoring epigraph data for duplicate triple: ('p4', ':instance', 'person')
ignoring epigraph data for duplicate triple: ('p4', ':wiki', '"Karkar"')
ignoring secondary node contexts for 'p4'
ignoring secondary node contexts for 'p5'
Smatch:  13%|█▎        | 22/165 [00:01<00:07, 18.10it/s]ignoring epigraph data for duplicate triple: ('c', ':instance', 'country')
ignoring epigraph data for duplicate triple: ('c', ':wiki', '"India"')
ignoring secondary node contexts for 'c'
Smatch:  21%|██        | 35/165 [00:01<00:06, 21.12it/s]cannot deinvert attribute: ('m', ':quant-of', 'inger')
cannot deinvert attribute: ('m2', ':quant-of', 'inger')
Smatch:  23%|██▎       | 38/165 [00:01<00:05, 22.97it/s


Total test examples : 165
Scored (parsed)     : 135
Skipped             : 30

--- Scores on 135 parseable pairs ---
Avg Precision   : 0.3294
Avg Recall      : 0.3512
Avg F1          : 0.3315

--- Treating skipped as F1=0 (n=165) ---
Avg F1 (all)    : 0.2712

Top-10 F1:
  [0098] F1=0.780  पुनर्रचणूक 31 ऑक्टोबर 2019 सावन लागू जाली.
  [0143] F1=0.680  नूर्दिन आनी पिकॉकाक 1996 च्या नोव्हेंबरांत बुकर ग्रुपान
  [0121] F1=0.670  सप्टेंबर 2014 वर्सा हो चित्रपट प्रदर्शीत जावपाचो आशिल्ल
  [0160] F1=0.650  अमिसाचो जल्म 17 जून 1922 दिसा इंग्लंडांत लंडनांतल्या  ड
  [0085] F1=0.610  2021 वर्सा भारतांतल्या निवळ शारांमदीं चंडीगढ 66व्या क्र
  [0026] F1=0.560  १२९६ त अलाउद्दीनान देवगिरीचेर आक्रमण केलें, आनी जलालुद्
  [0035] F1=0.520  फॅक्सॉन, ओक्लहोमा  फॅक्सॉन हें अमेरिकेंतल्या ओक्लहोमाचे
  [0113] F1=0.520  ताका वैजकी पर्यटनाचो एक प्रकार म्हूण मानूं येता.
  [0138] F1=0.520  2022 वर्सा प्रदर्शीत जावपाचें आशिल्ल्या त्रिशा स्टारर फ
  [0156] F1=0.510  पूण तुर्कस्तानांत उंटाचो बळी दिवप सामकें चुक्कून, आनी ह